# CE+ArcFace 20:80 diagnostic training

This notebook trains the **CE+ArcFace 20:80** ConvNeXtV2 checkpoints used in the post-challenge representation and controlled post-processing analyses.

- Training objective: `0.2 × cross-entropy loss + 0.8 × ArcFace loss`.
- Checkpoint selection and prediction: equal-weight fusion of the CE and margin-free ArcFace logits.
- Five-fold stratified cross-validation, early-stopping patience 4, and four deterministic TTA views are retained from the original run.
- This diagnostic model was trained after the challenge and did not affect the official sixth-place ranking.
- The official CE+ArcFace 60:40 branch is reproduced separately in `dlmmdd_solution_reproduction.ipynb`.

This is a post-challenge cleaned version of `arcface_base_ce20arc80.ipynb`. The local comparison cell, unused model branches, duplicate code, and absolute paths were removed while retaining the CE+ArcFace 20:80 experimental behavior.


## 1. Imports

In [ ]:
import os
import gc
import cv2
import math
import time
import json
import random
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import albumentations as A
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

## 2. Reproduction settings

The training-loss weights and the inference-head weights are intentionally defined separately. The original experiment used 20:80 for training and 50:50 for checkpoint selection and inference.


In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    display_name: str
    output_prefix: str
    train_ce_weight: float
    train_arc_weight: float
    infer_ce_weight: float
    infer_arc_weight: float


EXPERIMENT = ExperimentConfig(
    display_name="CE+ArcFace (20:80)",
    output_prefix="arcface_base_ce20arc80",
    train_ce_weight=0.2,
    train_arc_weight=0.8,
    infer_ce_weight=0.5,
    infer_arc_weight=0.5,
)


@dataclass
class CFG:
    seed: int = 42
    num_classes: int = 10

    # Paths are resolved from the repository root below.
    data_dir_name: str = "Data"
    train_csv_name: str = "training.csv"
    test_csv_name: str = "test.csv"
    artifact_dir_name: str = "ensemble_artifacts/diagnostics"

    model_name: str = "timm/convnextv2_base.fcmae_ft_in22k_in1k_384"
    image_size: int = 384
    embedding_dim: int = 512

    n_splits: int = 5
    epochs: int = 20

    train_bs: int = 16
    valid_bs: int = 32
    num_workers: int = 0

    lr_backbone: float = 1e-4
    lr_head: float = 2e-4
    min_lr: float = 1e-6
    weight_decay: float = 1e-4

    label_smoothing: float = 0.03
    dropout: float = 0.2

    arc_s: float = 30.0
    arc_m: float = 0.30

    patience: int = 4
    min_delta: float = 1e-4

    use_amp: bool = True
    grad_clip: float = 1.0

    tta_count: int = 4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


cfg = CFG()


def validate_experiment_settings():
    if not np.isclose(
        EXPERIMENT.train_ce_weight + EXPERIMENT.train_arc_weight,
        1.0,
    ):
        raise ValueError("Training-loss weights must sum to 1.")

    if not np.isclose(
        EXPERIMENT.infer_ce_weight + EXPERIMENT.infer_arc_weight,
        1.0,
    ):
        raise ValueError("Inference-head weights must sum to 1.")


validate_experiment_settings()

print("Device:", cfg.device)
print("Experiment:", EXPERIMENT.display_name)
print(
    "Training-loss weights:",
    EXPERIMENT.train_ce_weight,
    EXPERIMENT.train_arc_weight,
)
print(
    "Inference-head weights:",
    EXPERIMENT.infer_ce_weight,
    EXPERIMENT.infer_arc_weight,
)


## 3. Reproducibility utilities, repository paths, and data loading

The notebook can be started from the repository root or from a nested notebook directory. It searches upward for the repository marker files and then constructs all paths relative to that root.


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_repository_root(start_path):
    current = Path(start_path).resolve()

    while True:
        has_readme = (current / "README.md").exists()
        has_data_dir = (current / cfg.data_dir_name).exists()

        if has_readme and has_data_dir:
            return current

        if current == current.parent:
            break

        current = current.parent

    raise FileNotFoundError(
        "Repository root could not be located. "
        "Run the notebook inside the cloned repository."
    )


REPO_ROOT = find_repository_root(Path.cwd())
DATA_ROOT = REPO_ROOT / cfg.data_dir_name
TRAIN_CSV = DATA_ROOT / cfg.train_csv_name
TEST_CSV = DATA_ROOT / cfg.test_csv_name
SAVE_DIR = REPO_ROOT / cfg.artifact_dir_name
SAVE_DIR.mkdir(parents=True, exist_ok=True)


def make_abs_path(path_value):
    path = Path(str(path_value))
    if path.is_absolute():
        return str(path)
    return str(REPO_ROOT / path)


def validate_input_files():
    required = [TRAIN_CSV, TEST_CSV]
    missing = [path for path in required if not path.exists()]

    if missing:
        missing_text = "\n".join(f"- {path}" for path in missing)
        raise FileNotFoundError(
            "Required CSV file(s) were not found:\n"
            f"{missing_text}"
        )


set_seed(cfg.seed)
validate_input_files()

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

if "y" in train_df.columns:
    train_df["label"] = train_df["y"].astype(int)
elif "TARGET" in train_df.columns:
    train_df["label"] = train_df["TARGET"].astype(int)
else:
    raise ValueError("training.csv must contain either 'y' or 'TARGET'.")

required_train_columns = {"path", "label"}
required_test_columns = {"path", "ID"}

missing_train_columns = required_train_columns - set(train_df.columns)
missing_test_columns = required_test_columns - set(test_df.columns)

if missing_train_columns:
    raise ValueError(
        f"Missing training columns: {sorted(missing_train_columns)}"
    )
if missing_test_columns:
    raise ValueError(
        f"Missing test columns: {sorted(missing_test_columns)}"
    )

train_df["filepath"] = train_df["path"].apply(make_abs_path)
test_df["filepath"] = test_df["path"].apply(make_abs_path)

print("Repository root:", REPO_ROOT)
print("Artifact directory:", SAVE_DIR)
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nClass counts:")
print(train_df["label"].value_counts().sort_index())


## 4. Image transformations

The augmentation and four deterministic TTA views are unchanged from the original diagnostic runs.

In [ ]:
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def get_train_transforms(img_size=384):
    return A.Compose([
        A.RandomResizedCrop(
            size=(img_size, img_size),
            scale=(0.75, 1.0),
            ratio=(0.90, 1.10),
            p=1.0,
        ),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, border_mode=cv2.BORDER_REFLECT_101, p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.20),
        A.MotionBlur(blur_limit=(3, 5), p=0.10),
        A.RandomBrightnessContrast(
            brightness_limit=0.12,
            contrast_limit=0.12,
            p=0.35,
        ),
        A.Sharpen(
            alpha=(0.1, 0.25),
            lightness=(0.9, 1.1),
            p=0.15,
        ),
        A.ImageCompression(quality_range=(40, 100), p=0.35),
        A.ToGray(p=0.07),
        A.OneOf([
            A.Downscale(scale_range=(0.5, 0.85), p=1.0),
            A.Resize(int(img_size * 0.85), int(img_size * 0.85), p=1.0),
        ], p=0.12),
        A.Resize(img_size, img_size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


def get_valid_transforms(img_size=384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])


def get_tta_transforms(img_size=384, tta_id=0):
    if tta_id == 0:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])

    if tta_id == 1:
        return A.Compose([
            A.HorizontalFlip(p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])

    if tta_id == 2:
        crop_size = int(img_size * 0.92)
        return A.Compose([
            A.Resize(img_size, img_size),
            A.CenterCrop(height=crop_size, width=crop_size, p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])

    return A.Compose([
        A.Resize(int(img_size * 1.05), int(img_size * 1.05)),
        A.CenterCrop(height=img_size, width=img_size, p=1.0),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

## 5. Dataset

In [ ]:
class BaseImageDataset(Dataset):
    def __init__(self, dataframe, transform=None, is_test=False):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = cv2.imread(row["filepath"])
        if image is None:
            raise FileNotFoundError(row["filepath"])

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        if self.is_test:
            return image, int(row["ID"])

        return image, int(row["label"])

## 6. ConvNeXtV2 encoder, CE head, and ArcFace head

In [ ]:
class EarlyStopping:
    def __init__(self, patience=4, min_delta=1e-4, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return False

        if self.mode == "max":
            improved = score > self.best_score + self.min_delta
        else:
            improved = score < self.best_score - self.min_delta

        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.should_stop = True

        return self.should_stop


class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.30):
        super().__init__()
        self.s = s
        self.m = m

        self.weight = nn.Parameter(
            torch.FloatTensor(out_features, in_features)
        )
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels=None):
        cosine = F.linear(
            F.normalize(embeddings),
            F.normalize(self.weight),
        ).clamp(-1.0, 1.0)

        # During inference, return margin-free scaled cosine logits.
        if labels is None:
            return cosine * self.s

        sine = torch.sqrt(
            torch.clamp(1.0 - cosine.pow(2), min=1e-7)
        )
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(
            cosine > self.th,
            phi,
            cosine - self.mm,
        )

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1.0)

        logits = one_hot * phi + (1.0 - one_hot) * cosine
        return logits * self.s


class ArcFaceModel(nn.Module):
    def __init__(
        self,
        model_name,
        num_classes,
        embedding_dim=512,
        arc_s=30.0,
        arc_m=0.30,
        dropout=0.2,
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            global_pool="avg",
        )
        backbone_out = self.backbone.num_features

        self.neck = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.PReLU(),
        )
        self.dropout = nn.Dropout(dropout)

        self.ce_head = nn.Linear(embedding_dim, num_classes)
        self.arc_head = ArcMarginProduct(
            embedding_dim,
            num_classes,
            s=arc_s,
            m=arc_m,
        )

    def forward(self, x, labels=None):
        features = self.backbone(x)
        embeddings = self.neck(features)
        embeddings = self.dropout(embeddings)

        ce_logits = self.ce_head(embeddings)
        arc_logits = self.arc_head(embeddings, labels)

        return ce_logits, arc_logits, embeddings

## 7. Training and validation functions

The training objective is `0.2 × CE + 0.8 × ArcFace`. Validation accuracy for checkpoint selection uses a 50:50 fusion of the CE logits and the label-conditioned ArcFace logits, matching the original training run. TTA inference uses a 50:50 fusion of the CE logits and margin-free scaled cosine logits.


In [ ]:
CE_CRITERION = nn.CrossEntropyLoss(
    label_smoothing=cfg.label_smoothing
)
ARC_CRITERION = nn.CrossEntropyLoss()


def fuse_logits(ce_logits, arc_logits, experiment):
    return (
        experiment.infer_ce_weight * ce_logits
        + experiment.infer_arc_weight * arc_logits
    )


def train_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    device,
    experiment,
):
    model.train()

    scaler = torch.cuda.amp.GradScaler(
        enabled=cfg.use_amp
    )

    total_loss = 0.0
    total_ce = 0.0
    total_arc = 0.0
    total_correct = 0
    total_count = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels)
            fused_logits = fuse_logits(
                ce_logits,
                arc_logits,
                experiment,
            )

            ce_loss = CE_CRITERION(ce_logits, labels)
            arc_loss = ARC_CRITERION(arc_logits, labels)

            loss = (
                experiment.train_ce_weight * ce_loss
                + experiment.train_arc_weight * arc_loss
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            cfg.grad_clip,
        )
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_ce += ce_loss.item() * batch_size
        total_arc += arc_loss.item() * batch_size
        total_correct += (
            fused_logits.argmax(dim=1) == labels
        ).sum().item()
        total_count += batch_size

    return (
        total_loss / total_count,
        total_ce / total_count,
        total_arc / total_count,
        total_correct / total_count,
    )


@torch.no_grad()
def validate_one_epoch(
    model,
    loader,
    device,
    experiment,
):
    model.eval()

    total_loss = 0.0
    total_ce = 0.0
    total_arc = 0.0
    total_correct = 0
    total_count = 0
    all_probs = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(images, labels)
            fused_logits = fuse_logits(
                ce_logits,
                arc_logits,
                experiment,
            )
            probs = torch.softmax(fused_logits, dim=1)

            ce_loss = CE_CRITERION(ce_logits, labels)
            arc_loss = ARC_CRITERION(arc_logits, labels)
            loss = (
                experiment.train_ce_weight * ce_loss
                + experiment.train_arc_weight * arc_loss
            )

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_ce += ce_loss.item() * batch_size
        total_arc += arc_loss.item() * batch_size
        total_correct += (
            probs.argmax(dim=1) == labels
        ).sum().item()
        total_count += batch_size

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    return (
        total_loss / total_count,
        total_ce / total_count,
        total_arc / total_count,
        total_correct / total_count,
        np.concatenate(all_probs),
        np.concatenate(all_labels),
    )

## 8. Deterministic four-view TTA

In [ ]:
@torch.no_grad()
def predict_test_probs(
    model,
    loader,
    device,
    experiment,
):
    model.eval()

    all_probs = []
    all_ids = []

    for images, ids in loader:
        images = images.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=cfg.use_amp):
            ce_logits, arc_logits, _ = model(
                images,
                labels=None,
            )
            logits = fuse_logits(
                ce_logits,
                arc_logits,
                experiment,
            )
            probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_ids.extend(
            ids.tolist()
            if torch.is_tensor(ids)
            else list(ids)
        )

    return (
        np.asarray(all_ids),
        np.concatenate(all_probs, axis=0),
    )


@torch.no_grad()
def predict_with_tta(
    model,
    dataframe,
    device,
    experiment,
    is_test=True,
    tta_count=4,
):
    probs_list = []
    labels_ref = None
    ids_ref = None

    for tta_id in range(tta_count):
        dataset = BaseImageDataset(
            dataframe,
            transform=get_tta_transforms(
                cfg.image_size,
                tta_id,
            ),
            is_test=is_test,
        )
        loader = DataLoader(
            dataset,
            batch_size=cfg.valid_bs,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=True,
            drop_last=False,
        )

        if is_test:
            ids, probs = predict_test_probs(
                model,
                loader,
                device,
                experiment,
            )
            ids_ref = ids
            probs_list.append(probs)
        else:
            fold_probs = []
            fold_labels = []

            for images, labels in loader:
                images = images.to(
                    device,
                    non_blocking=True,
                )

                with torch.cuda.amp.autocast(
                    enabled=cfg.use_amp
                ):
                    ce_logits, arc_logits, _ = model(
                        images,
                        labels=None,
                    )
                    logits = fuse_logits(
                        ce_logits,
                        arc_logits,
                        experiment,
                    )
                    probs = torch.softmax(
                        logits,
                        dim=1,
                    )

                fold_probs.append(
                    probs.cpu().numpy()
                )
                fold_labels.append(
                    labels.numpy()
                )

            labels_ref = np.concatenate(
                fold_labels,
                axis=0,
            )
            probs_list.append(
                np.concatenate(
                    fold_probs,
                    axis=0,
                )
            )

        del dataset, loader
        gc.collect()

    mean_probs = np.mean(
        probs_list,
        axis=0,
    )

    if is_test:
        return ids_ref, mean_probs

    return mean_probs, labels_ref

## 9. Cross-validation runner and output files

In [ ]:
def output_path(experiment, suffix):
    return SAVE_DIR / (
        f"{experiment.output_prefix}_{suffix}"
    )


def best_model_path(experiment, fold):
    return output_path(
        experiment,
        f"best_fold{fold}.pth",
    )


def run_experiment(
    experiment,
    train_dataframe,
    test_dataframe,
):

    print("\n" + "#" * 90)
    print(f"EXPERIMENT: {experiment.display_name}")
    print(
        "Training loss weights:"
        f" CE={experiment.train_ce_weight:.1f},"
        f" ArcFace={experiment.train_arc_weight:.1f}"
    )
    print(
        "Inference head weights:"
        f" CE={experiment.infer_ce_weight:.1f},"
        f" ArcFace={experiment.infer_arc_weight:.1f}"
    )
    print("#" * 90)

    # Reset the random state before cross-validation.
    set_seed(cfg.seed)

    splitter = StratifiedKFold(
        n_splits=cfg.n_splits,
        shuffle=True,
        random_state=cfg.seed,
    )

    oof_probs = np.zeros(
        (len(train_dataframe), cfg.num_classes),
        dtype=np.float32,
    )
    test_probs = np.zeros(
        (len(test_dataframe), cfg.num_classes),
        dtype=np.float32,
    )
    fold_scores = []

    experiment_start = time.time()

    for fold, (train_indices, valid_indices) in enumerate(
        splitter.split(
            train_dataframe,
            train_dataframe["label"],
        )
    ):
        print("\n" + "=" * 90)
        print(
            f"{experiment.display_name} | "
            f"FOLD {fold + 1}/{cfg.n_splits}"
        )
        print("=" * 90)

        train_fold_df = train_dataframe.iloc[
            train_indices
        ].reset_index(drop=True)
        valid_fold_df = train_dataframe.iloc[
            valid_indices
        ].reset_index(drop=True)

        train_dataset = BaseImageDataset(
            train_fold_df,
            transform=get_train_transforms(
                cfg.image_size
            ),
            is_test=False,
        )
        valid_dataset = BaseImageDataset(
            valid_fold_df,
            transform=get_valid_transforms(
                cfg.image_size
            ),
            is_test=False,
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=cfg.train_bs,
            shuffle=True,
            num_workers=cfg.num_workers,
            pin_memory=True,
        )
        valid_loader = DataLoader(
            valid_dataset,
            batch_size=cfg.valid_bs,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=True,
        )

        model = ArcFaceModel(
            model_name=cfg.model_name,
            num_classes=cfg.num_classes,
            embedding_dim=cfg.embedding_dim,
            arc_s=cfg.arc_s,
            arc_m=cfg.arc_m,
            dropout=cfg.dropout,
        ).to(cfg.device)

        backbone_params = list(
            model.backbone.parameters()
        )
        head_params = (
            list(model.neck.parameters())
            + list(model.ce_head.parameters())
            + list(model.arc_head.parameters())
        )

        optimizer = torch.optim.AdamW(
            [
                {
                    "params": backbone_params,
                    "lr": cfg.lr_backbone,
                },
                {
                    "params": head_params,
                    "lr": cfg.lr_head,
                },
            ],
            weight_decay=cfg.weight_decay,
        )
        scheduler = (
            torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=cfg.epochs * len(train_loader),
                eta_min=cfg.min_lr,
            )
        )

        stopper = EarlyStopping(
            patience=cfg.patience,
            min_delta=cfg.min_delta,
            mode="max",
        )
        best_acc = -1.0
        checkpoint_path = best_model_path(
            experiment,
            fold,
        )

        for epoch in range(cfg.epochs):
            epoch_start = time.time()

            train_metrics = train_one_epoch(
                model,
                train_loader,
                optimizer,
                scheduler,
                cfg.device,
                experiment,
            )
            valid_metrics = validate_one_epoch(
                model,
                valid_loader,
                cfg.device,
                experiment,
            )

            (
                train_loss,
                train_ce,
                train_arc,
                train_acc,
            ) = train_metrics
            (
                valid_loss,
                valid_ce,
                valid_arc,
                valid_acc,
                _,
                _,
            ) = valid_metrics

            print(
                f"Epoch {epoch + 1:02d}/{cfg.epochs} | "
                f"train_loss={train_loss:.4f} "
                f"(ce={train_ce:.4f}, arc={train_arc:.4f}) | "
                f"train_acc={train_acc:.4f} | "
                f"valid_loss={valid_loss:.4f} "
                f"(ce={valid_ce:.4f}, arc={valid_arc:.4f}) | "
                f"valid_acc={valid_acc:.4f} | "
                f"time={time.time() - epoch_start:.1f}s"
            )

            if valid_acc > best_acc:
                best_acc = valid_acc
                torch.save(
                    model.state_dict(),
                    checkpoint_path,
                )
                print(
                    f"  saved best -> "
                    f"{checkpoint_path}"
                )

            if stopper.step(valid_acc):
                print(
                    "  early stopping at "
                    f"epoch {epoch + 1}"
                )
                break

        model.load_state_dict(
            torch.load(
                checkpoint_path,
                map_location=cfg.device,
            )
        )

        valid_tta_probs, valid_tta_labels = (
            predict_with_tta(
                model,
                valid_fold_df,
                cfg.device,
                experiment,
                is_test=False,
                tta_count=cfg.tta_count,
            )
        )
        valid_tta_acc = accuracy_score(
            valid_tta_labels,
            valid_tta_probs.argmax(axis=1),
        )

        print(
            f"Fold {fold + 1} TTA valid acc: "
            f"{valid_tta_acc:.5f}"
        )

        oof_probs[valid_indices] = valid_tta_probs
        np.save(
            output_path(
                experiment,
                f"fold{fold}_val_probs.npy",
            ),
            valid_tta_probs,
        )

        _, fold_test_probs = predict_with_tta(
            model,
            test_dataframe,
            cfg.device,
            experiment,
            is_test=True,
            tta_count=cfg.tta_count,
        )
        test_probs += (
            fold_test_probs / cfg.n_splits
        )
        np.save(
            output_path(
                experiment,
                f"fold{fold}_test_probs.npy",
            ),
            fold_test_probs,
        )

        fold_scores.append({
            "fold": fold,
            "tta_valid_acc": float(
                valid_tta_acc
            ),
            "best_valid_acc": float(best_acc),
        })

        del (
            model,
            optimizer,
            scheduler,
            train_loader,
            valid_loader,
            train_dataset,
            valid_dataset,
        )
        gc.collect()
        torch.cuda.empty_cache()

    oof_predictions = oof_probs.argmax(axis=1)
    oof_accuracy = accuracy_score(
        train_dataframe["label"].values,
        oof_predictions,
    )

    print("\n" + "=" * 90)
    print(
        f"{experiment.display_name} "
        f"OOF ACCURACY: {oof_accuracy:.6f}"
    )
    print("=" * 90)

    np.save(
        output_path(
            experiment,
            "oof_probs.npy",
        ),
        oof_probs,
    )
    np.save(
        output_path(
            experiment,
            "test_probs.npy",
        ),
        test_probs,
    )

    submission = test_dataframe[["ID"]].copy()
    submission["TARGET"] = (
        test_probs.argmax(axis=1).astype(int)
    )
    submission.to_csv(
        output_path(
            experiment,
            "submission.csv",
        ),
        index=False,
    )

    fold_scores_df = pd.DataFrame(
        fold_scores
    )
    fold_scores_df.to_csv(
        output_path(
            experiment,
            "fold_scores.csv",
        ),
        index=False,
    )

    metadata = {
        "experiment_name": "ce20arc80",
        "display_name": experiment.display_name,
        "output_prefix": experiment.output_prefix,
        "experiment": asdict(experiment),
        "cfg": asdict(cfg),
        "oof_accuracy": float(oof_accuracy),
        "n_train": int(len(train_dataframe)),
        "n_test": int(len(test_dataframe)),
        "elapsed_seconds": float(
            time.time() - experiment_start
        ),
    }
    with open(
        output_path(
            experiment,
            "metadata.json",
        ),
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print(
        "Saved checkpoint pattern:",
        output_path(
            experiment,
            "best_fold{fold}.pth",
        ),
    )
    print(
        "Saved OOF probabilities:",
        output_path(
            experiment,
            "oof_probs.npy",
        ),
    )
    print(
        "Saved test probabilities:",
        output_path(
            experiment,
            "test_probs.npy",
        ),
    )
    print(
        "Saved submission:",
        output_path(
            experiment,
            "submission.csv",
        ),
    )
    print(
        "Saved fold scores:",
        output_path(
            experiment,
            "fold_scores.csv",
        ),
    )
    print(
        "Saved metadata:",
        output_path(
            experiment,
            "metadata.json",
        ),
    )

    return {
        "experiment_name": "ce20arc80",
        "display_name": experiment.display_name,
        "oof_accuracy": float(oof_accuracy),
        "mean_tta_valid_accuracy": float(
            fold_scores_df[
                "tta_valid_acc"
            ].mean()
        ),
        "elapsed_seconds": float(
            time.time() - experiment_start
        ),
    }

## 10. Run the CE+ArcFace 20:80 experiment

This cell trains all five folds, reloads the best checkpoint from each fold, performs four-view TTA, and writes the OOF/test probabilities and metadata. It can take a long time on a single GPU.


In [ ]:
summary = run_experiment(
    EXPERIMENT,
    train_df,
    test_df,
)

pd.DataFrame([summary])


## Expected output files

The notebook writes the following files under `ensemble_artifacts/diagnostics/`:

```text
arcface_base_ce20arc80_best_fold0.pth
arcface_base_ce20arc80_best_fold1.pth
arcface_base_ce20arc80_best_fold2.pth
arcface_base_ce20arc80_best_fold3.pth
arcface_base_ce20arc80_best_fold4.pth
arcface_base_ce20arc80_fold0_val_probs.npy
arcface_base_ce20arc80_fold0_test_probs.npy
...
arcface_base_ce20arc80_oof_probs.npy
arcface_base_ce20arc80_test_probs.npy
arcface_base_ce20arc80_submission.csv
arcface_base_ce20arc80_fold_scores.csv
arcface_base_ce20arc80_metadata.json
```

The checkpoint prefix matches the files consumed by the representation and controlled post-processing analysis notebooks.
